## Set up

In [ ]:
# set scripts

import subprocess
import sys
from pathlib import Path


ROOT = Path.cwd().resolve()

def run_script(module_name):
    subprocess.run(
        [
            sys.executable,
            "-m",
            f"scripts.{module_name}"
        ],
        cwd=ROOT,
        check=True
    )

print("Project root:", ROOT)
print("Python executable:", sys.executable)

In [ ]:
# check packages

import csv
import json
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import pickle
import requests
import shap
import tqdm
from scipy.stats import hypergeom, combine_pvalues
from sklearn import metrics
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.svm import SVC
from xgboost import XGBClassifier

from src.features import select_features
from src.training import train_model_repeated_stratified_rf

print("All imports successful")

In [ ]:
# combine PDB inputs

from pathlib import Path

def merge_feature_matrix(input_files, output_file):
    output_path = Path(output_file)

    with output_path.open("wb") as outfile:
        for i, input_file in enumerate(input_files):
            input_path = Path(input_file)

            with input_path.open("rb") as infile:
                if i > 0:
                    infile.readline()  # 略過 header

                for line in infile:
                    outfile.write(line)

    size_mb = output_path.stat().st_size / (1024 * 1024)
    print(f"Merged: {output_path.name} ({size_mb:.2f} MiB)")

    return str(output_path)

merge_feature_matrix(
    [
        "./data/pdb_valid_HAMW_filtered_part1.txt",
        "./data/pdb_valid_HAMW_filtered_part2.txt",
        "./data/pdb_valid_HAMW_filtered_part3.txt"
    ],
    "./data/pdb_valid_HAMW_filtered.txt"
)

## Training and prediction

In [ ]:
run_script("compare_feature_sets")

In [ ]:
run_script("compare_models")

In [ ]:
run_script("train_final_models")

In [ ]:
run_script("screen_pdb")

## Data processing and analysis

In [ ]:
# check features

import pandas as pd
from src.features import select_features

for stage in ["first", "second"]:
    data = pd.read_csv(
        f"./data/{stage}_1048.txt",
        sep="\t",
        index_col=0
    )

    X = data.iloc[:, 1:]

    for feature_set in ["1048", "ecfp", "moiety", "numeric"]:
        selected = select_features(X, feature_set)
        print(stage, feature_set, selected.shape, "\n")

In [ ]:
# calculate pairing proteins and pairs using given ligand list 

run_script("get_ligands_proteins_pairs_number")

In [ ]:
# draw SHAP beeswarm plot

from src.interpretation import draw_shap
draw_shap(
    input_dataset_path="./data/first_1048.txt",
    out_of_fold_shap_mean_path=(
        "./results/rf_first_1048/"
        "out_of_fold_SHAP_mean.csv"
    ),
    max_display_num=21
)

from src.interpretation import draw_shap
draw_shap(
    input_dataset_path="./data/second_1048.txt",
    out_of_fold_shap_mean_path=(
        "./results/rf_second_1048/"
        "out_of_fold_SHAP_mean.csv"
    ),
    max_display_num=21
)

In [ ]:
# perform hypergeom test

run_script("analyze_sglt2_aa_enrichment")

## Compound features

In [ ]:
%load_ext autoreload
%autoreload 2

from src.features import (
    mol_to_ecfp,
    mol_to_checkmol,
    mol_to_pubchem,
    mol_to_ringsindrugs,
    mol_to_ac,
    mol_to_stereo,
    mol_to_smiles,
    smiles_to_stereo,
    smiles_to_temp_mol
    )

In [ ]:
print(mol_to_stereo(smiles_to_temp_mol('CCc1ccc(Cc2ccc3CO[C@]4(O[C@H](CO)[C@@H](O)[C@H](O)[C@@H]4O)c3c2)cc1')))